# PoWR + ATLAS/Phoenix fitting to Gaia BP/RP spectra
# v1 - With Priors from A23 and W25 Dust Map

**Changes from v0:**
- **NEW**: A_V prior from Wang+2025 3D dust map (configurable weight)
- **NEW**: Teff prior from Andrae+2023 for cool stars (Teff_A23 < 7500K)
- Uses enriched catalog from `retrieve_BPRP_spectra-export_2026.v1.ipynb`
- Output includes which priors were applied

**Prior formulation:**
```
chi2_total = chi2_spectral + chi2_parallax + w_AV * chi2_AV_prior + w_Teff * chi2_Teff_prior
```

**Input:**
- `model_manifest.csv` (from download notebook)
- Enriched catalog FITS file (with Teff_A23, A_V_W25 columns)
- `./BPRP_spectra/<source_id>.fits` (XP_SAMPLED format)

**Output:**
- CSV with all fit parameters plus prior information

In [1]:
!pip install dust_extinction

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from pathlib import Path
import pandas as pd
import glob
import re
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

# Check for dust_extinction package
try:
    from dust_extinction.parameter_averages import G23
    try:
        from dust_extinction.parameter_averages import G24
    except ImportError:
        G24 = None
    print("dust_extinction package loaded successfully")
except ImportError:
    raise ImportError(
        "Please install dust_extinction >= 1.2:\n"
        "    pip install dust_extinction\n"
    )

dust_extinction package loaded successfully


In [ ]:
# ========================== CONFIGURATION ==========================
POWR_MODEL_DIR   = Path('./griddl-gal-ob-vd3-line_calib')
ATLAS_MODEL_DIR  = Path('./stellar_models')
MODEL_MANIFEST   = Path('./model_manifest.csv')
CATALOG_FITS     = 'SB1Cands.3_enriched.fits'  # Use enriched catalog from v1 retrieval
SPECTRA_DIR      = Path('./BPRP_spectra')
PLOT_DIR         = Path('./fit_plots')
PLOT_DIR.mkdir(exist_ok=True)

# Auto-generate output CSV name from input catalog
OUTPUT_CSV       = Path(CATALOG_FITS).stem + '_fits.csv'  # e.g., SB1Cands.3_enriched_fits.csv
print(f"Output will be: {OUTPUT_CSV}")

# ============== TEST MODE ==============
N_MAX_FIT = None  # Set to None to fit all stars, or a number for testing

# ============== PRIOR CONFIGURATION ==============
# A_V prior from Wang+2025 dust map
USE_AV_PRIOR = True
AV_PRIOR_WEIGHT = 1.0  # Weight for A_V prior (1.0 = standard chi2)

# Teff prior from Andrae+2023 (only for cool stars)
USE_TEFF_PRIOR = True
TEFF_PRIOR_WEIGHT = 1.0  # Weight for Teff prior
TEFF_PRIOR_TEFF_MAX = 7500  # Only apply prior for stars with Teff_A23 < this
TEFF_PRIOR_SIGMA = 500  # K, uncertainty to assume for A23 Teff

# Fitting grid
A_V_GRID  = np.arange(0.0, 7.1, 0.1)
R_V_GRID  = np.array([2.5, 3.1, 3.7])
WAVELENGTH_FIT_MIN = 340.0
WAVELENGTH_FIT_MAX = 900.0
SYSTEMATIC_FLOOR   = 0.03
BLUE_WEIGHT_REGION = (340.0, 480.0)
BLUE_WEIGHT_FACTOR = 2.0

W_M2_NM_TO_ERG_S_CM2_A = 1e2

# Check manifest
USE_MANIFEST = MODEL_MANIFEST.exists()
if USE_MANIFEST:
    print(f"Model manifest found: {MODEL_MANIFEST}")
else:
    print(f"No manifest at {MODEL_MANIFEST} - will use glob-based loading")

# Report settings
print(f"\nPrior settings:")
print(f"  A_V prior: {'ON' if USE_AV_PRIOR else 'OFF'} (weight={AV_PRIOR_WEIGHT})")
print(f"  Teff prior: {'ON' if USE_TEFF_PRIOR else 'OFF'} (weight={TEFF_PRIOR_WEIGHT}, Teff_max={TEFF_PRIOR_TEFF_MAX}K, sigma={TEFF_PRIOR_SIGMA}K)")

if N_MAX_FIT is not None:
    print(f"\nTEST MODE: Will fit only first {N_MAX_FIT} stars")

In [4]:
# ========================== LOAD CATALOG ==========================
print("Loading catalog...")
with fits.open(CATALOG_FITS) as hdul:
    data = hdul[1].data
    source_ids       = data['source_id']
    gmag             = data['Gmag']
    parallax         = data['parallax'] if 'parallax' in data.names else data['Plx']
    parallax_error   = data['parallax_error'] if 'parallax_error' in data.names else np.full(len(source_ids), 0.1)
    
    # Load enrichment columns (may be NaN if not matched)
    Teff_A23_catalog = data['Teff_A23'] if 'Teff_A23' in data.names else np.full(len(source_ids), np.nan)
    logg_A23_catalog = data['logg_A23'] if 'logg_A23' in data.names else np.full(len(source_ids), np.nan)
    MH_A23_catalog   = data['MH_A23'] if 'MH_A23' in data.names else np.full(len(source_ids), np.nan)
    A_V_W25_catalog  = data['A_V_W25'] if 'A_V_W25' in data.names else np.full(len(source_ids), np.nan)
    A_V_W25_err_catalog = data['A_V_W25_err'] if 'A_V_W25_err' in data.names else np.full(len(source_ids), np.nan)

print(f"Loaded {len(source_ids)} sources")
print(f"\nEnrichment columns:")
print(f"  Teff_A23: {np.sum(np.isfinite(Teff_A23_catalog))} valid values")
print(f"  A_V_W25: {np.sum(np.isfinite(A_V_W25_catalog))} valid values")

# Count how many will get priors
n_av_prior = np.sum(np.isfinite(A_V_W25_catalog)) if USE_AV_PRIOR else 0
n_teff_prior = np.sum(np.isfinite(Teff_A23_catalog) & (Teff_A23_catalog < TEFF_PRIOR_TEFF_MAX)) if USE_TEFF_PRIOR else 0
print(f"\nStars eligible for priors:")
print(f"  A_V prior: {n_av_prior} stars")
print(f"  Teff prior: {n_teff_prior} stars (Teff_A23 < {TEFF_PRIOR_TEFF_MAX}K)")

Loading catalog...
Loaded 1258 sources

Enrichment columns:
  Teff_A23: 946 valid values
  A_V_W25: 1203 valid values

Stars eligible for priors:
  A_V prior: 1203 stars
  Teff prior: 946 stars (Teff_A23 < 7500K)


In [5]:
# ============================================================================
# LOAD ALL MODELS (from manifest if available, else glob)
# ============================================================================

all_models = []

if USE_MANIFEST:
    print(f"\nLoading models from manifest: {MODEL_MANIFEST}")
    
    manifest_df = pd.read_csv(MODEL_MANIFEST)
    print(f"  Manifest contains {len(manifest_df)} models")
    
    for src in manifest_df['source'].unique():
        subset = manifest_df[manifest_df['source'] == src]
        print(f"  {src}: {len(subset)} models, Teff={subset['Teff'].min()}-{subset['Teff'].max()}K")
    
    for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Loading models"):
        source = row['source']
        filename = row['filename']
        
        if source == 'PoWR':
            filepath = POWR_MODEL_DIR / filename
        else:
            filepath = ATLAS_MODEL_DIR / filename
        
        if not filepath.exists():
            continue
        
        try:
            data = np.loadtxt(filepath)
            wave_model = data[:, 0]
            log_flux_model = data[:, 1]
            flux_model = 10**log_flux_model
            
            all_models.append({
                'Teff': row['Teff'],
                'logg': row['logg'],
                'source': source,
                'logL': row['logL'],
                'Mass': row['Mass'],
                'Mass_std': row.get('Mass_std', np.nan),
                'R_Rsun': row['R_Rsun'],
                'wavelength': wave_model,
                'flux': flux_model,
                'filename': filename
            })
        except Exception as e:
            print(f"  Error loading {filepath}: {e}")
    
    print(f"\nSuccessfully loaded {len(all_models)} models from manifest")

else:
    # Fallback glob-based loading (same as v0)
    print("\nLoading models via glob (no manifest)...")
    # [glob loading code here - same as v0]
    raise NotImplementedError("Please run download notebook to generate model_manifest.csv")

powr_models = all_models
print(f"\nModel library ready: {len(powr_models)} models")


Loading models from manifest: model_manifest.csv
  Manifest contains 711 models
  ATLAS: 39 models, Teff=10000-15000K
  PHOENIX: 192 models, Teff=3800-10000K
  PoWR: 480 models, Teff=15000-56000K


Loading models: 100%|█████████████████████████████████████████████| 711/711 [00:11<00:00, 60.41it/s]


Successfully loaded 711 models from manifest

Model library ready: 711 models


In [6]:
# ========================== GAIA GRID & RESAMPLE ==========================
ref_file = next(SPECTRA_DIR.glob('*.fits'))
with fits.open(ref_file) as hdul:
    gaia_wavelength_nm = np.asarray(hdul[1].data['wavelength']).flatten()
gaia_wavelength_A = gaia_wavelength_nm * 10

diff = np.diff(gaia_wavelength_A)
bins = np.concatenate([[gaia_wavelength_A[0]-diff[0]/2],
                       gaia_wavelength_A[:-1] + diff/2,
                       [gaia_wavelength_A[-1]+diff[-1]/2]])

print("Resampling models to Gaia grid...")
for m in tqdm(powr_models):
    flux = np.zeros(len(gaia_wavelength_A))
    for i in range(len(gaia_wavelength_A)):
        mask = (m['wavelength'] >= bins[i]) & (m['wavelength'] < bins[i+1])
        if mask.sum()>0:
            flux[i] = np.mean(m['flux'][mask])
        else:
            flux[i] = np.interp(gaia_wavelength_A[i], m['wavelength'], m['flux'])
    flux = gaussian_filter1d(flux, sigma=2.0)
    m['flux_gaia'] = flux
    m['wave_gaia'] = gaia_wavelength_A

Resampling models to Gaia grid...


100%|█████████████████████████████████████████████████████████████| 711/711 [00:49<00:00, 14.42it/s]


In [7]:
# ============================================================================
# EXTINCTION FUNCTION - GORDON+ 2023
# ============================================================================

import astropy.units as u

if G24 is not None:
    ExtinctionModel = G24
    print("Using extinction model: G24 (Gordon+ 2024)")
else:
    ExtinctionModel = G23
    print("Using extinction model: G23 (Gordon+ 2023)")

def build_extinction_table(wavelength_aa, R_V_grid):
    """Pre-compute A(lambda)/A_V for all R_V values."""
    table = {}
    wave_with_units = wavelength_aa * u.AA
    for Rv in R_V_grid:
        ext = ExtinctionModel(Rv=Rv)
        table[Rv] = ext(wave_with_units)
    return table

print(f"Building {ExtinctionModel.__name__} extinction lookup table...")
EXT_TABLE = build_extinction_table(gaia_wavelength_A, R_V_GRID)
print(f"   Extinction table ready for R_V = {list(R_V_GRID)}")

def apply_reddening(wavelength_aa, flux, A_V, R_V=3.1):
    """Apply Gordon+ extinction to a spectrum."""
    if R_V in EXT_TABLE:
        A_lambda_over_Av = EXT_TABLE[R_V]
    else:
        ext = ExtinctionModel(Rv=R_V)
        A_lambda_over_Av = ext(wavelength_aa * u.AA)
    A_lambda = A_lambda_over_Av * A_V
    return flux * 10**(-0.4 * A_lambda)

def calculate_A_G(A_V, R_V=3.1):
    """Compute A_G (Gaia G band) from A_V."""
    lambda_G_aa = 6420.0
    ext = ExtinctionModel(Rv=R_V)
    A_lambda_over_Av = ext(lambda_G_aa * u.AA)
    return float(A_lambda_over_Av) * A_V

def wavelength_weights(wavelength_nm, blue_region=BLUE_WEIGHT_REGION, blue_weight=BLUE_WEIGHT_FACTOR):
    """Return weights for chi-square."""
    weights = np.ones_like(wavelength_nm)
    blue_mask = (wavelength_nm >= blue_region[0]) & (wavelength_nm <= blue_region[1])
    weights[blue_mask] = blue_weight
    return weights

Using extinction model: G23 (Gordon+ 2023)
Building G23 extinction lookup table...
   Extinction table ready for R_V = [np.float64(2.5), np.float64(3.1), np.float64(3.7)]


In [8]:
# ========================== FITTING FUNCTION WITH PRIORS ==========================

def fit_single_star(sid, plx, plx_err, gmag, 
                    Teff_A23=np.nan, A_V_W25=np.nan, A_V_W25_err=np.nan):
    """
    Fit a single star with optional priors.
    
    Parameters
    ----------
    sid : int
        Gaia source_id
    plx, plx_err : float
        Parallax and error in mas
    gmag : float
        G magnitude
    Teff_A23 : float
        Andrae+2023 Teff (NaN if not available)
    A_V_W25, A_V_W25_err : float
        Wang+2025 A_V and error (NaN if not available)
    
    Returns
    -------
    dict with fit results
    """
    file = SPECTRA_DIR / f"{sid}.fits"
    if not file.exists():
        return None
    
    with fits.open(file) as h:
        d = h[1].data
        w_nm = d['wavelength'].flatten()
        f    = d['flux'].flatten() * 1e2
        err  = d['flux_error'].flatten() * 1e2 if 'flux_error' in d.names else f*0.01
    
    err = np.sqrt(err**2 + (SYSTEMATIC_FLOOR*f)**2)
    w_A = w_nm * 10
    mask = (w_A >= 3400) & (w_A <= 9000)
    if mask.sum() < 10:
        return None
    
    # Determine which priors to apply
    use_av_prior = USE_AV_PRIOR and np.isfinite(A_V_W25) and np.isfinite(A_V_W25_err) and A_V_W25_err > 0
    use_teff_prior = USE_TEFF_PRIOR and np.isfinite(Teff_A23) and Teff_A23 < TEFF_PRIOR_TEFF_MAX
    
    # Set prior uncertainties
    sigma_av = A_V_W25_err if use_av_prior else 1.0  # dummy if not used
    sigma_teff = TEFF_PRIOR_SIGMA
    
    def evaluate_av(av_val):
        """Find best chi2 across all models and Rv values for a given Av"""
        best_chi2_for_av = np.inf
        best_for_av = None
        
        for m in powr_models:
            # Teff prior: skip models far from A23 Teff if prior is active
            chi2_teff = 0.0
            if use_teff_prior:
                chi2_teff = TEFF_PRIOR_WEIGHT * ((m['Teff'] - Teff_A23) / sigma_teff)**2
            
            for Rv in R_V_GRID:
                fmod = apply_reddening(m['wave_gaia'], m['flux_gaia'], av_val, Rv)
                scale = np.sum(f[mask] * fmod[mask] / err[mask]**2) / np.sum(fmod[mask]**2 / err[mask]**2)
                dist = 10.0 / np.sqrt(scale) if scale > 0 else np.inf
                
                # Spectral chi2
                chi_spec = np.sum(wavelength_weights(w_nm[mask]) * ((f[mask] - scale*fmod[mask]) / err[mask])**2)
                
                # Parallax chi2
                chi_plx = ((1000/dist - plx) / plx_err)**2 if (plx > 0 and plx_err > 0 and np.isfinite(plx_err) and np.isfinite(dist)) else 0
                
                # A_V prior chi2
                chi2_av = 0.0
                if use_av_prior:
                    chi2_av = AV_PRIOR_WEIGHT * ((av_val - A_V_W25) / sigma_av)**2
                
                # Total chi2
                chi_tot = chi_spec + chi_plx + chi2_av + chi2_teff
                
                if chi_tot < best_chi2_for_av:
                    best_chi2_for_av = chi_tot
                    best_for_av = {
                        **m, 
                        'A_V': av_val, 
                        'R_V': Rv, 
                        'scale': scale, 
                        'dist_pc': dist,
                        'chi2_tot': chi_tot, 
                        'chi2_spec': chi_spec, 
                        'chi2_plx': chi_plx,
                        'chi2_av_prior': chi2_av,
                        'chi2_teff_prior': chi2_teff,
                        'used_av_prior': use_av_prior,
                        'used_teff_prior': use_teff_prior,
                        'flux_mod': fmod * scale, 
                        'w_obs': w_nm, 
                        'f_obs': f, 
                        'err_obs': err
                    }
        
        return best_chi2_for_av, best_for_av
    
    # Stage 1: Coarse grid search
    av_coarse = np.arange(0, 6.05, 0.5)
    chi2_coarse = []
    results_coarse = []
    
    for av_val in av_coarse:
        chi2, result = evaluate_av(av_val)
        chi2_coarse.append(chi2)
        results_coarse.append(result)
    
    chi2_coarse = np.array(chi2_coarse)
    best_coarse_idx = np.argmin(chi2_coarse)
    
    # Stage 2: Refine with iterations
    av_center = av_coarse[best_coarse_idx]
    search_radius = 0.5
    
    for iteration in range(3):
        av_refined = np.linspace(max(0, av_center - search_radius), 
                                 min(6.0, av_center + search_radius), 5)
        chi2_refined = []
        results_refined = []
        
        for av_val in av_refined:
            chi2, result = evaluate_av(av_val)
            chi2_refined.append(chi2)
            results_refined.append(result)
        
        chi2_refined = np.array(chi2_refined)
        best_refined_idx = np.argmin(chi2_refined)
        
        # Parabolic fit
        if 0 < best_refined_idx < len(av_refined) - 1:
            idx_range = [best_refined_idx - 1, best_refined_idx, best_refined_idx + 1]
            av_pts = av_refined[idx_range]
            chi_pts = chi2_refined[idx_range]
            try:
                coeffs = np.polyfit(av_pts, chi_pts, 2)
                a, b, c = coeffs
                if a > 0:
                    av_parabolic = -b / (2 * a)
                    if av_pts[0] <= av_parabolic <= av_pts[2]:
                        av_center = av_parabolic
                    else:
                        av_center = av_refined[best_refined_idx]
                else:
                    av_center = av_refined[best_refined_idx]
            except:
                av_center = av_refined[best_refined_idx]
        else:
            av_center = av_refined[best_refined_idx]
        
        av_center = np.clip(av_center, 0, 6.0)
        search_radius = search_radius / 2.5
    
    # Final evaluation
    chi2_final, best_result = evaluate_av(av_center)
    
    if chi2_coarse[best_coarse_idx] < chi2_final:
        return results_coarse[best_coarse_idx]
    
    return best_result

In [ ]:
# ========================== PLOTTING FUNCTION ==========================
def plot_fit(sid, fit_params, Teff_A23=np.nan, A_V_W25=np.nan):
    """Generate diagnostic plot for one star."""
    if fit_params is None:
        return

    w_nm  = fit_params['w_obs']
    f_obs = fit_params['f_obs']
    f_mod = fit_params['flux_mod']
    err   = fit_params['err_obs']

    w_A   = w_nm * 10.0
    mask  = (w_A >= 3400) & (w_A <= 9000)

    dist_pc = fit_params['dist_pc']
    dist_str = f"{dist_pc:.0f}" if np.isfinite(dist_pc) and dist_pc > 0 else "???"

    n_spec = mask.sum()
    n_plx  = 1 if fit_params.get('chi2_plx', 0) > 0 else 0
    dof    = n_spec + n_plx - 1
    chi2_red = fit_params['chi2_tot'] / dof if dof > 0 else 999.99

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8),
                                   gridspec_kw={'height_ratios': [3, 1]})

    ax1.plot(w_nm, f_obs, 'ko', ms=3, alpha=0.7, label='Gaia BP/RP')
    
    # Build label with fit parameters
    label = f"Teff={fit_params['Teff']/1000:.0f} kK  log g={fit_params['logg']:.2f}\n"
    label += f"A_V={fit_params['A_V']:.2f}  R_V={fit_params['R_V']:.1f}"
    
    # Add A23/W25 values whenever available (not just when used as prior)
    prior_parts = []
    if np.isfinite(A_V_W25):
        prior_parts.append(f"$A_V$(W25)={A_V_W25:.2f}")
    if np.isfinite(Teff_A23):
        prior_parts.append(f"$T_{{eff}}$(A23)={Teff_A23:.0f}K")
    if prior_parts:
        label += "\n" + "  ".join(prior_parts)
    
    ax1.plot(w_nm, f_mod, 'r-', lw=2, label=label)

    # Rayleigh-Jeans reference
    w_ref = 600.0
    f_ref = np.interp(w_ref, w_nm, f_mod, left=np.nan, right=np.nan)
    if np.isfinite(f_ref):
        f_ref *= 0.9
        w_rj = np.linspace(340, 1050, 200)
        ax1.plot(w_rj, f_ref * (w_ref/w_rj)**4, '--', color='blue', lw=1.5,
                 alpha=0.8, label='Rayleigh-Jeans')

    ax1.set_yscale('log')
    ax1.set_xlim(330, 1050)
    ax1.set_ylabel('Flux (erg s$^{-1}$ cm$^{-2}$ A$^{-1}$)')
    ax1.legend(fontsize=10)
    ax1.set_title(f"source_id = {sid} | d = {dist_str} pc | chi2_red = {chi2_red:.2f}")

    resid = (f_obs - f_mod) / err
    ax2.plot(w_nm, resid, 'ko', ms=3, alpha=0.7)
    ax2.axhline(0, color='red', ls='--', lw=1)
    ax2.axhspan(-3, 3, color='gray', alpha=0.1)
    ax2.set_xlabel('Wavelength (nm)')
    ax2.set_ylabel('Residual (sigma)')
    ax2.set_xlim(330, 1050)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    outfile = PLOT_DIR / f"fit_{sid}.png"
    plt.savefig(outfile, dpi=150, bbox_inches='tight')
    plt.close()

In [ ]:
# ========================== MAIN LOOP ==========================

n_to_fit = len(source_ids) if N_MAX_FIT is None else min(N_MAX_FIT, len(source_ids))
print(f"Fitting {n_to_fit} stars" + (" (TEST MODE)" if N_MAX_FIT else "") + "\n")

results = []
n_with_av_prior = 0
n_with_teff_prior = 0

for i, sid in enumerate(source_ids[:n_to_fit], 1):
    print(f"[{i:4d}/{n_to_fit}] {sid} ", end='')
    
    # Get prior values for this star
    teff_a23 = Teff_A23_catalog[i-1]
    av_w25 = A_V_W25_catalog[i-1]
    av_w25_err = A_V_W25_err_catalog[i-1]
    
    res = fit_single_star(sid, parallax[i-1], parallax_error[i-1], gmag[i-1],
                          Teff_A23=teff_a23, A_V_W25=av_w25, A_V_W25_err=av_w25_err)
    
    if not res:
        print("no spectrum")
        continue
    
    A_G = calculate_A_G(res['A_V'], res['R_V'])
    M_G = gmag[i-1] - 5*np.log10(res['dist_pc']) + 5 - A_G if np.isfinite(res['dist_pc']) else np.nan
    
    # Track prior usage
    if res.get('used_av_prior', False):
        n_with_av_prior += 1
    if res.get('used_teff_prior', False):
        n_with_teff_prior += 1
    
    result_dict = {
        'source_id': sid,
        'Teff': res['Teff'],
        'logg': res['logg'],
        'A_V': res['A_V'],
        'R_V': res['R_V'],
        'distance_pc': res['dist_pc'],
        'gmag': gmag[i-1],
        'M_G': M_G,
        'chi2_red': res['chi2_tot']/(len(res['w_obs'])-1),
        'model_source': res.get('source', 'unknown'),
        'model_filename': res.get('filename', ''),
        'logL': res.get('logL', np.nan),
        'Mass': res.get('Mass', np.nan),
        'Mass_std': res.get('Mass_std', np.nan),
        'R_Rsun': res.get('R_Rsun', np.nan),
        # Prior info
        'Teff_A23': teff_a23,
        'A_V_W25': av_w25,
        'used_av_prior': res.get('used_av_prior', False),
        'used_teff_prior': res.get('used_teff_prior', False),
        'chi2_av_prior': res.get('chi2_av_prior', 0),
        'chi2_teff_prior': res.get('chi2_teff_prior', 0),
    }
    results.append(result_dict)
    
    # Plot with prior values for annotation
    plot_fit(sid, res, Teff_A23=teff_a23, A_V_W25=av_w25)
    
    # Print summary
    prior_str = ""
    if res.get('used_av_prior', False):
        prior_str += f" [AV:{av_w25:.2f}]"
    if res.get('used_teff_prior', False):
        prior_str += f" [Teff:{teff_a23:.0f}]"
    mass_str = f" M={res.get('Mass', np.nan):.1f}Msun" if np.isfinite(res.get('Mass', np.nan)) else ""
    print(f"Teff={res['Teff']/1000:.0f}kK logg={res['logg']:.1f} A_V={res['A_V']:.2f} d={res['dist_pc']:.0f}pc{mass_str}{prior_str}")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nFinished! Saved to {OUTPUT_CSV}")
print(f"Fitted {len(results)} stars successfully")
print(f"\nPrior usage:")
print(f"  A_V prior applied: {n_with_av_prior} stars")
print(f"  Teff prior applied: {n_with_teff_prior} stars")
print(f"\nOutput columns: {list(df.columns)}")

In [11]:
# Summary statistics
print("="*60)
print("FIT SUMMARY")
print("="*60)
print(f"\nTotal stars fitted: {len(df)}")
print(f"\nTeff distribution:")
print(f"  Min: {df['Teff'].min():.0f} K")
print(f"  Max: {df['Teff'].max():.0f} K")
print(f"  Median: {df['Teff'].median():.0f} K")
print(f"\nA_V distribution:")
print(f"  Min: {df['A_V'].min():.2f} mag")
print(f"  Max: {df['A_V'].max():.2f} mag")
print(f"  Median: {df['A_V'].median():.2f} mag")
print(f"\nModel sources:")
print(df['model_source'].value_counts())
print(f"\nPriors applied:")
print(f"  A_V prior: {df['used_av_prior'].sum()} ({100*df['used_av_prior'].mean():.1f}%)")
print(f"  Teff prior: {df['used_teff_prior'].sum()} ({100*df['used_teff_prior'].mean():.1f}%)")

FIT SUMMARY

Total stars fitted: 18

Teff distribution:
  Min: 5400 K
  Max: 33000 K
  Median: 11000 K

A_V distribution:
  Min: 0.73 mag
  Max: 6.00 mag
  Median: 3.19 mag

Model sources:
model_source
ATLAS      10
PHOENIX     6
PoWR        2
Name: count, dtype: int64

Priors applied:
  A_V prior: 18 (100.0%)
  Teff prior: 12 (66.7%)
